# Supplier Intelligence Copilot — RAG

This notebook builds the qualitative intelligence layer of the Supplier Intelligence Copilot.

The RAG pipeline uses supplier review notes and incident records to answer questions such as:

- What issues have been reported for a supplier?
- Why has a supplier's performance deteriorated?
- What corrective actions were recommended?
- What incidents have occurred recently?
- What recurring operational problems appear in supplier reviews?

Structured KPI questions will continue to be handled through SQL.

Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('d:/Supplier-Intelligence-Copilot')

Connect to the database

In [2]:
from src.database import get_connection

conn = get_connection()

Inspect our text tables

In [3]:
reviews = pd.read_sql_query(
    """
    SELECT *
    FROM supplier_reviews
    LIMIT 5;
    """,
    conn
)

reviews

,review_id,supplier_id,review_date,review_type,performance_summary,key_issues,corrective_actions,reviewer_notes
0,REV00001,SUP001,2025-01-31,Monthly Performance Review,Supplier performance remained broadly stable d...,SLA compliance is below the preferred operatin...,"Increase operating capacity, strengthen qualit...",Priority should remain on trend monitoring and...
1,REV00002,SUP001,2025-02-28,Monthly Performance Review,Supplier performance improved during the recen...,SLA compliance is below the preferred operatin...,Supplier to submit a 30-day corrective action ...,Recent improvements should be validated over t...
2,REV00003,SUP001,2025-03-31,Monthly Performance Review,Supplier performance remained broadly stable d...,SLA compliance is below the preferred operatin...,Supplier to submit a 30-day corrective action ...,Priority should remain on trend monitoring and...
3,REV00004,SUP001,2025-04-30,Monthly Performance Review,Supplier performance remained broadly stable d...,SLA compliance is below the preferred operatin...,Supplier to submit a 30-day corrective action ...,Priority should remain on trend monitoring and...
4,REV00005,SUP001,2025-05-31,Monthly Performance Review,Supplier performance remained broadly stable d...,SLA compliance is below the preferred operatin...,Introduce additional process controls and moni...,Priority should remain on trend monitoring and...


In [4]:
reviews.columns.tolist()

['review_id',
 'supplier_id',
 'review_date',
 'review_type',
 'performance_summary',
 'key_issues',
 'corrective_actions',
 'reviewer_notes']

Do the same for Incidents

In [5]:
incidents = pd.read_sql_query(
    """
    SELECT *
    FROM incidents
    LIMIT 5;
    """,
    conn
)

incidents

,incident_id,supplier_id,incident_date,severity,incident_type,description,resolution
0,INC00001,SUP001,2025-01-01,Medium,Capacity Issue,Supplier capacity was insufficient for expecte...,The issue was escalated to supplier leadership...
1,INC00002,SUP001,2025-01-02,High,SLA Breach,Supplier performance fell below the agreed SLA...,An improvement plan was agreed with fortnightl...
2,INC00003,SUP001,2025-02-02,Medium,Delivery Delay,"Repeated delivery delays were observed, with m...",An improvement plan was agreed with fortnightl...
3,INC00004,SUP001,2025-02-15,Medium,Delivery Delay,"Repeated delivery delays were observed, with m...",Additional staffing and management oversight w...
4,INC00005,SUP001,2025-02-20,Medium,Communication Breakdown,Escalation handling was delayed because of com...,Supplier committed to weekly status updates an...


In [6]:
incidents.columns.tolist()

['incident_id',
 'supplier_id',
 'incident_date',
 'severity',
 'incident_type',
 'description',
 'resolution']

Load supplier reviews

In [7]:
reviews_df = pd.read_sql_query(
    """
    SELECT
        r.review_id,
        r.supplier_id,
        s.supplier_name,
        r.review_date,
        r.performance_summary,
        r.key_issues,
        r.corrective_actions,
        r.reviewer_notes

    FROM supplier_reviews r

    JOIN suppliers s
        ON r.supplier_id = s.supplier_id;
    """,
    conn
)

reviews_df.head()

,review_id,supplier_id,supplier_name,review_date,performance_summary,key_issues,corrective_actions,reviewer_notes
0,REV00001,SUP001,BluePeak Solutions,2025-01-31,Supplier performance remained broadly stable d...,SLA compliance is below the preferred operatin...,"Increase operating capacity, strengthen qualit...",Priority should remain on trend monitoring and...
1,REV00002,SUP001,BluePeak Solutions,2025-02-28,Supplier performance improved during the recen...,SLA compliance is below the preferred operatin...,Supplier to submit a 30-day corrective action ...,Recent improvements should be validated over t...
2,REV00003,SUP001,BluePeak Solutions,2025-03-31,Supplier performance remained broadly stable d...,SLA compliance is below the preferred operatin...,Supplier to submit a 30-day corrective action ...,Priority should remain on trend monitoring and...
3,REV00004,SUP001,BluePeak Solutions,2025-04-30,Supplier performance remained broadly stable d...,SLA compliance is below the preferred operatin...,Supplier to submit a 30-day corrective action ...,Priority should remain on trend monitoring and...
4,REV00005,SUP001,BluePeak Solutions,2025-05-31,Supplier performance remained broadly stable d...,SLA compliance is below the preferred operatin...,Introduce additional process controls and moni...,Priority should remain on trend monitoring and...


In [8]:
reviews_df.shape

(1245, 8)

Load incidents

In [ ]:
incidents_df = pd.read_sql_query(
    """
    SELECT
        i.incident_id,
        i.supplier_id,
        s.supplier_name,
        i.incident_date,
        i.incident_type,
        i.severity,
        i.description,
        i.resolution

    FROM incidents i

    JOIN suppliers s
        ON i.supplier_id = s.supplier_id;
    """,
    conn
)

incidents_df.head()

,incident_id,supplier_id,supplier_name,incident_date,incident_type,severity,description,resolution
0,INC00001,SUP001,BluePeak Solutions,2025-01-01,Capacity Issue,Medium,Supplier capacity was insufficient for expecte...,The issue was escalated to supplier leadership...
1,INC00002,SUP001,BluePeak Solutions,2025-01-02,SLA Breach,High,Supplier performance fell below the agreed SLA...,An improvement plan was agreed with fortnightl...
2,INC00003,SUP001,BluePeak Solutions,2025-02-02,Delivery Delay,Medium,"Repeated delivery delays were observed, with m...",An improvement plan was agreed with fortnightl...
3,INC00004,SUP001,BluePeak Solutions,2025-02-15,Delivery Delay,Medium,"Repeated delivery delays were observed, with m...",Additional staffing and management oversight w...
4,INC00005,SUP001,BluePeak Solutions,2025-02-20,Communication Breakdown,Medium,Escalation handling was delayed because of com...,Supplier committed to weekly status updates an...


In [10]:
incidents_df.shape

(1370, 8)

Convert them into RAG documents

Reviews

In [11]:
review_documents = []

for _, row in reviews_df.iterrows():

    text = f"""
Supplier: {row['supplier_name']}
Review Date: {row['review_date']}

Performance Summary:
{row['performance_summary']}

Key Issues:
{row['key_issues']}

Corrective Actions:
{row['corrective_actions']}

Reviewer Notes:
{row['reviewer_notes']}
""".strip()

    review_documents.append({
        "id": f"review_{row['review_id']}",
        "text": text,
        "supplier_id": row["supplier_id"],
        "supplier_name": row["supplier_name"],
        "document_type": "review",
        "date": row["review_date"]
    })

Incidents

In [12]:
incident_documents = []

for _, row in incidents_df.iterrows():

    text = f"""
Supplier: {row['supplier_name']}
Incident Date: {row['incident_date']}
Incident Type: {row['incident_type']}
Severity: {row['severity']}

Incident:
{row['description']}

Resolution:
{row['resolution']}
""".strip()

    incident_documents.append({
        "id": f"incident_{row['incident_id']}",
        "text": text,
        "supplier_id": row["supplier_id"],
        "supplier_name": row["supplier_name"],
        "document_type": "incident",
        "date": row["incident_date"]
    })

Combine both

In [13]:
documents = (
    review_documents
    + incident_documents
)

len(documents)

2615

Generate embeddings

In [14]:
from sentence_transformers import SentenceTransformer

In [15]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Get Text

In [16]:
texts = [
    document["text"]
    for document in documents
]

Generate Embeddings

In [17]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/82 [00:00<?, ?it/s]

In [18]:
embeddings.shape

(2615, 384)

Store in ChromaDB

In [19]:
import chromadb

VECTOR_DB_PATH = (
    PROJECT_ROOT
    / "vector_store"
)

VECTOR_DB_PATH.mkdir(
    parents=True,
    exist_ok=True
)

Create Client

In [20]:
client = chromadb.PersistentClient(
    path=str(VECTOR_DB_PATH)
)

To make notebook reruns easy

In [21]:
try:
    client.delete_collection(
        name="supplier_intelligence"
    )
except Exception:
    pass

Create Collection

In [31]:
collection = client.create_collection(
    name="supplier_intelligence"
)

InternalError: Collection [supplier_intelligence] already exists

Prepare Metadata

In [23]:
ids = [
    document["id"]
    for document in documents
]

metadatas = [
    {
        "supplier_id": str(document["supplier_id"]),
        "supplier_name": document["supplier_name"],
        "document_type": document["document_type"],
        "date": str(document["date"])
    }
    for document in documents
]

In [24]:
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

In [25]:
collection.count()

2615

Build retrieval

In [38]:
def retrieve_documents(
    query,
    top_k=5,
    supplier_name=None,
    max_per_supplier=2
):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    where_filter = None

    if supplier_name:
        where_filter = {
            "supplier_name": supplier_name
        }

    # Retrieve more candidates than needed
    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k * 4,
        where=where_filter
    )

    retrieved = []

    supplier_counts = {}

    for i in range(
        len(results["documents"][0])
    ):

        metadata = (
            results["metadatas"][0][i]
        )

        supplier = (
            metadata["supplier_name"]
        )

        supplier_counts.setdefault(
            supplier,
            0
        )

        if (
            supplier_counts[supplier]
            >= max_per_supplier
        ):
            continue

        retrieved.append({
            "document":
                results["documents"][0][i],

            "metadata":
                metadata,

            "distance":
                results["distances"][0][i]
        })

        supplier_counts[supplier] += 1

        if len(retrieved) == top_k:
            break

    return retrieved

Test retrieval

In [39]:
results = retrieve_documents(
    "What supplier issues are causing poor delivery performance?",
    top_k=5
)

for result in results:
    print(
        result["metadata"]
    )
    print(
        result["document"]
    )
    print("-" * 80)

{'document_type': 'review', 'supplier_id': 'SUP052', 'date': '2025-05-31', 'supplier_name': 'Apex Global'}
Supplier: Apex Global
Review Date: 2025-05-31

Performance Summary:
Supplier performance deteriorated during the recent review period. SLA compliance is 93.7%, on-time delivery is 90.0%, and defect rate is 2.7%.

Key Issues:
on-time delivery performance is below expectations. Review discussion highlighted quality-control process gaps as a likely contributing factor.

Corrective Actions:
Introduce additional process controls and monitor SLA, delivery, and defect metrics each week.

Reviewer Notes:
Priority should remain on trend monitoring and early intervention.
--------------------------------------------------------------------------------
{'supplier_id': 'SUP052', 'document_type': 'incident', 'supplier_name': 'Apex Global', 'date': '2026-02-26'}
Supplier: Apex Global
Incident Date: 2026-02-26
Incident Type: Delivery Delay
Severity: Low

Incident:
Repeated delivery delays were o

In [40]:
results = retrieve_documents(
    "What recurring quality problems have suppliers experienced?",
    top_k=5
)

In [41]:
results = retrieve_documents(
    "Which incidents involved delayed delivery or logistics problems?",
    top_k=5
)

At this stage, don't add the LLM yet.

First verify that semantic retrieval itself makes sense.

Calling the LLM using API

In [33]:
import os

from dotenv import load_dotenv
from openai import OpenAI


load_dotenv(
    PROJECT_ROOT / ".env"
)

OPENROUTER_API_KEY = os.getenv(
    "OPENROUTER_API_KEY"
)

MODEL_NAME = os.getenv(
    "OPENROUTER_MODEL",
    "openai/gpt-oss-20b:free"
)


if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY not found in .env"
    )


llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

Build the RAG context

In [34]:
def build_context(retrieved_documents):

    context_blocks = []

    for i, result in enumerate(
        retrieved_documents,
        start=1
    ):

        metadata = result["metadata"]
        document = result["document"]

        block = f"""
SOURCE {i}
Supplier: {metadata['supplier_name']}
Document Type: {metadata['document_type']}
Date: {metadata['date']}

{document}
""".strip()

        context_blocks.append(block)

    return "\n\n".join(
        context_blocks
    )

Create our RAG answer function

In [44]:
def ask_supplier_rag(
    question,
    top_k=5,
    supplier_name=None
):

    # Step 1: Semantic retrieval
    retrieved = retrieve_documents(
        query=question,
        top_k=top_k,
        supplier_name=supplier_name
    )

    # Step 2: Build context
    context = build_context(
        retrieved
    )

    # Step 3: Grounded prompt
    system_prompt = """
You are a Supplier Intelligence Copilot.

Your role is to help procurement and vendor-management
teams understand qualitative supplier issues.

Use ONLY the supplied review and incident evidence.

Rules:
You are a Supplier Intelligence Copilot.

Your role is to help procurement and vendor-management
teams understand qualitative supplier issues.

Use ONLY the supplied review and incident evidence.

Rules:
1. Do not invent supplier issues.
2. Do not invent metrics or financial values.
3. If the evidence is insufficient, clearly say so.
4. Focus on recurring issues, incidents, corrective actions,
   and operational patterns.
5. Keep the response concise and business-oriented.
6. Mention whether evidence comes from supplier reviews,
   incidents, or both.
7. Do not generalize findings to the entire supplier base.
   Describe conclusions as patterns observed in the
   retrieved evidence.
"""

    user_prompt = f"""
QUESTION:

{question}


RETRIEVED SUPPLIER EVIDENCE:

{context}


Based only on the evidence above, answer the question.
"""

    # Step 4: LLM call
    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.2,
        max_tokens=500
    )

    answer = (
        response
        .choices[0]
        .message
        .content
    )

    return answer, retrieved

Test It

In [45]:
question = """
What recurring supplier issues are causing
delivery performance problems?
"""

answer, sources = ask_supplier_rag(
    question,
    top_k=5
)

print(answer)

**Recurring supplier issues driving delivery performance problems**

| Issue | Evidence Source | Supplier | Incident Type | Key Detail |
|-------|-----------------|----------|---------------|------------|
| **Quality‑control process gaps** | Source 1 & 2 (Apex Global incidents) | Apex Global | Delivery Delay | Repeated delays attributed to QC gaps; improvement plan with fortnightly checkpoints |
| **Forecasting & planning gaps** | Source 3 (Apex Systems incident) | Apex Systems | Delivery Delay | Repeated delays linked to forecasting/planning deficiencies; strengthened controls and monitoring |
| **Subcontractor performance issues** | Source 4 & 5 (Sterling Systems incidents) | Sterling Systems | Delivery Delay | Repeated delays tied to subcontractor performance; escalation to leadership and control strengthening |

**Summary**

The evidence shows that delivery performance problems are consistently linked to three main recurring issues:

1. **Quality‑control process gaps** (Apex Global

Inspect what the LLM actually saw

In [46]:
for i, source in enumerate(
    sources,
    start=1
):

    print(
        f"\nSOURCE {i}"
    )

    print(
        source["metadata"]
    )

    print(
        source["document"]
    )

    print(
        "-" * 80
    )


SOURCE 1
{'document_type': 'incident', 'date': '2026-01-14', 'supplier_name': 'Apex Global', 'supplier_id': 'SUP052'}
Supplier: Apex Global
Incident Date: 2026-01-14
Incident Type: Delivery Delay
Severity: High

Incident:
Repeated delivery delays were observed, with quality-control process gaps cited as a contributing factor.

Resolution:
An improvement plan was agreed with fortnightly performance checkpoints.
--------------------------------------------------------------------------------

SOURCE 2
{'supplier_id': 'SUP052', 'date': '2026-05-03', 'document_type': 'incident', 'supplier_name': 'Apex Global'}
Supplier: Apex Global
Incident Date: 2026-05-03
Incident Type: Delivery Delay
Severity: Medium

Incident:
Repeated delivery delays were observed, with quality-control process gaps cited as a contributing factor.

Resolution:
An improvement plan was agreed with fortnightly performance checkpoints.
--------------------------------------------------------------------------------

SOURC

SQL supplier intelligence function

In [47]:
def get_supplier_sql_context(supplier_name):

    query = """
    WITH supplier_spend AS (

        SELECT
            supplier_id,
            SUM(invoice_amount) AS total_spend,
            AVG(payment_delay_days) AS avg_payment_delay,

            100.0
            * SUM(invoice_error_flag)
            / COUNT(*) AS invoice_error_rate

        FROM invoices

        GROUP BY supplier_id
    ),

    supplier_performance AS (

        SELECT
            supplier_id,

            AVG(sla_compliance) AS avg_sla,
            AVG(on_time_delivery_rate) AS avg_delivery,
            AVG(defect_rate) AS avg_defect_rate,
            AVG(invoice_accuracy) AS avg_invoice_accuracy,
            AVG(avg_resolution_days) AS avg_resolution_days,

            SUM(escalation_count) AS total_escalations

        FROM monthly_performance

        GROUP BY supplier_id
    ),

    incident_summary AS (

        SELECT
            supplier_id,
            COUNT(*) AS incident_count,

            SUM(
                CASE
                    WHEN severity IN ('High', 'Critical')
                    THEN 1
                    ELSE 0
                END
            ) AS severe_incidents

        FROM incidents

        GROUP BY supplier_id
    )

    SELECT
        s.supplier_id,
        s.supplier_name,
        s.category,
        s.region,
        s.criticality,

        ROUND(sp.total_spend, 2)
            AS total_spend,

        ROUND(p.avg_sla, 2)
            AS avg_sla,

        ROUND(p.avg_delivery, 2)
            AS avg_delivery,

        ROUND(p.avg_defect_rate, 2)
            AS avg_defect_rate,

        ROUND(p.avg_invoice_accuracy, 2)
            AS avg_invoice_accuracy,

        ROUND(p.avg_resolution_days, 2)
            AS avg_resolution_days,

        p.total_escalations,

        ROUND(sp.invoice_error_rate, 2)
            AS invoice_error_rate,

        COALESCE(i.incident_count, 0)
            AS incident_count,

        COALESCE(i.severe_incidents, 0)
            AS severe_incidents

    FROM suppliers s

    LEFT JOIN supplier_spend sp
        ON s.supplier_id = sp.supplier_id

    LEFT JOIN supplier_performance p
        ON s.supplier_id = p.supplier_id

    LEFT JOIN incident_summary i
        ON s.supplier_id = i.supplier_id

    WHERE s.supplier_name = ?;
    """

    return pd.read_sql_query(
        query,
        conn,
        params=[supplier_name]
    )

In [48]:
get_supplier_sql_context(
    "Apex Global"
)

,supplier_id,supplier_name,category,region,criticality,total_spend,avg_sla,avg_delivery,avg_defect_rate,avg_invoice_accuracy,avg_resolution_days,total_escalations,invoice_error_rate,incident_count,severe_incidents
0,SUP052,Apex Global,Hardware,North,High,33324697.76,90.26,89.44,3.22,95.16,3.13,29,3.26,25,7


Add recent trend analysis

In [49]:
def get_supplier_trend(supplier_name):

    query = """
    WITH supplier_months AS (

        SELECT
            s.supplier_name,
            p.month,
            p.sla_compliance,
            p.on_time_delivery_rate,
            p.defect_rate,
            p.escalation_count,

            ROW_NUMBER() OVER (
                ORDER BY p.month DESC
            ) AS month_rank

        FROM monthly_performance p

        JOIN suppliers s
            ON p.supplier_id = s.supplier_id

        WHERE s.supplier_name = ?
    )

    SELECT

        ROUND(
            AVG(
                CASE
                    WHEN month_rank BETWEEN 1 AND 3
                    THEN sla_compliance
                END
            ),
            2
        ) AS recent_3m_sla,

        ROUND(
            AVG(
                CASE
                    WHEN month_rank BETWEEN 4 AND 6
                    THEN sla_compliance
                END
            ),
            2
        ) AS previous_3m_sla,


        ROUND(
            AVG(
                CASE
                    WHEN month_rank BETWEEN 1 AND 3
                    THEN on_time_delivery_rate
                END
            ),
            2
        ) AS recent_3m_delivery,

        ROUND(
            AVG(
                CASE
                    WHEN month_rank BETWEEN 4 AND 6
                    THEN on_time_delivery_rate
                END
            ),
            2
        ) AS previous_3m_delivery,


        ROUND(
            AVG(
                CASE
                    WHEN month_rank BETWEEN 1 AND 3
                    THEN defect_rate
                END
            ),
            2
        ) AS recent_3m_defect,

        ROUND(
            AVG(
                CASE
                    WHEN month_rank BETWEEN 4 AND 6
                    THEN defect_rate
                END
            ),
            2
        ) AS previous_3m_defect

    FROM supplier_months;
    """

    return pd.read_sql_query(
        query,
        conn,
        params=[supplier_name]
    )

In [50]:
get_supplier_trend(
    "Apex Global"
)

,recent_3m_sla,previous_3m_sla,recent_3m_delivery,previous_3m_delivery,recent_3m_defect,previous_3m_defect
0,86.84,88.3,86.44,88.85,3.75,4.02


Turn SQL output into LLM-friendly text

In [51]:
def build_sql_context(
    supplier_name
):

    metrics_df = get_supplier_sql_context(
        supplier_name
    )

    trend_df = get_supplier_trend(
        supplier_name
    )

    if metrics_df.empty:
        return None

    metrics = (
        metrics_df.iloc[0]
    )

    trend = (
        trend_df.iloc[0]
    )

    context = f"""
SUPPLIER STRUCTURED METRICS

Supplier: {metrics['supplier_name']}
Category: {metrics['category']}
Region: {metrics['region']}
Criticality: {metrics['criticality']}

Overall Performance:
- Average SLA: {metrics['avg_sla']}%
- Average on-time delivery: {metrics['avg_delivery']}%
- Average defect rate: {metrics['avg_defect_rate']}%
- Average invoice accuracy: {metrics['avg_invoice_accuracy']}%
- Average resolution time: {metrics['avg_resolution_days']} days
- Total escalations: {metrics['total_escalations']}

Financial / Invoice:
- Total spend: {metrics['total_spend']}
- Invoice error rate: {metrics['invoice_error_rate']}%

Incidents:
- Total incidents: {metrics['incident_count']}
- Severe incidents: {metrics['severe_incidents']}

Recent Trend:

SLA:
- Latest 3 months: {trend['recent_3m_sla']}%
- Previous 3 months: {trend['previous_3m_sla']}%

Delivery:
- Latest 3 months: {trend['recent_3m_delivery']}%
- Previous 3 months: {trend['previous_3m_delivery']}%

Defect Rate:
- Latest 3 months: {trend['recent_3m_defect']}%
- Previous 3 months: {trend['previous_3m_defect']}%
""".strip()

    return context

In [52]:
print(
    build_sql_context(
        "Apex Global"
    )
)

SUPPLIER STRUCTURED METRICS

Supplier: Apex Global
Category: Hardware
Region: North
Criticality: High

Overall Performance:
- Average SLA: 90.26%
- Average on-time delivery: 89.44%
- Average defect rate: 3.22%
- Average invoice accuracy: 95.16%
- Average resolution time: 3.13 days
- Total escalations: 29

Financial / Invoice:
- Total spend: 33324697.76
- Invoice error rate: 3.26%

Incidents:
- Total incidents: 25
- Severe incidents: 7

Recent Trend:

SLA:
- Latest 3 months: 86.84%
- Previous 3 months: 88.3%

Delivery:
- Latest 3 months: 86.44%
- Previous 3 months: 88.85%

Defect Rate:
- Latest 3 months: 3.75%
- Previous 3 months: 4.02%


Build hybrid copilot

In [53]:
def ask_hybrid_copilot(
    question,
    supplier_name,
    top_k=5
):

    # -------------------------
    # 1. SQL context
    # -------------------------

    sql_context = build_sql_context(
        supplier_name
    )

    if sql_context is None:
        return (
            f"Supplier '{supplier_name}' "
            "was not found."
        )

    # -------------------------
    # 2. RAG retrieval
    # -------------------------

    retrieved = retrieve_documents(
        query=question,
        supplier_name=supplier_name,
        top_k=top_k,
        max_per_supplier=top_k
    )

    rag_context = build_context(
        retrieved
    )

    # -------------------------
    # 3. Grounded prompt
    # -------------------------

    system_prompt = """
You are a Supplier Intelligence Copilot for procurement
and vendor-management teams.

You have two evidence sources:

1. STRUCTURED SQL METRICS
   These contain quantitative supplier KPIs and trends.

2. RETRIEVED QUALITATIVE EVIDENCE
   These contain supplier reviews and incident records.

Use only the supplied evidence.

Rules:

1. Never invent numbers, incidents, issues or causes.
2. Clearly distinguish quantitative observations from
   qualitative explanations.
3. Use recent KPI trends when discussing deterioration.
4. Use supplier reviews and incidents to explain possible
   operational drivers.
5. Do not claim causation unless the retrieved evidence
   explicitly supports it.
6. If qualitative evidence only suggests an explanation,
   describe it as a possible contributing factor.
7. Keep the answer concise and business-oriented.
8. End with 1-2 practical procurement actions when supported
   by the evidence.
"""

    user_prompt = f"""
QUESTION:

{question}


STRUCTURED SQL EVIDENCE:

{sql_context}


RETRIEVED REVIEW / INCIDENT EVIDENCE:

{rag_context}


Provide a grounded supplier intelligence answer.
"""

    # -------------------------
    # 4. Call LLM
    # -------------------------

    response = (
        llm_client
        .chat
        .completions
        .create(
            model=MODEL_NAME,

            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],

            temperature=0.2,
            max_tokens=600
        )
    )

    answer = (
        response
        .choices[0]
        .message
        .content
    )

    return {
        "answer": answer,
        "sql_context": sql_context,
        "sources": retrieved
    }

Test the full hybrid system

In [54]:
result = ask_hybrid_copilot(
    question="""
    Why is Apex Global performing poorly,
    and what should procurement focus on?
    """,
    supplier_name="Apex Global"
)

print(
    result["answer"]
)

**Why Apex Global is under‑performing**

| Quantitative trend (latest 3 months) | Observation |
|-------------------------------------|-------------|
| SLA compliance 86.84 % (↓ 1.46 pp from previous 3 months) | Falling below the preferred operating threshold. |
| On‑time delivery 86.44 % (↓ 2.41 pp) | Consistently below expectations. |
| Defect rate 3.75 % (↑ 0.27 pp) | Higher than the previous period, despite a slight improvement over the 4.02 % earlier. |
| Incidents 25, severe 7 |


Show evidence separately

In [55]:
print(
    "========== SQL EVIDENCE ==========\n"
)

print(
    result["sql_context"]
)

========== SQL EVIDENCE ==========

SUPPLIER STRUCTURED METRICS

Supplier: Apex Global
Category: Hardware
Region: North
Criticality: High

Overall Performance:
- Average SLA: 90.26%
- Average on-time delivery: 89.44%
- Average defect rate: 3.22%
- Average invoice accuracy: 95.16%
- Average resolution time: 3.13 days
- Total escalations: 29

Financial / Invoice:
- Total spend: 33324697.76
- Invoice error rate: 3.26%

Incidents:
- Total incidents: 25
- Severe incidents: 7

Recent Trend:

SLA:
- Latest 3 months: 86.84%
- Previous 3 months: 88.3%

Delivery:
- Latest 3 months: 86.44%
- Previous 3 months: 88.85%

Defect Rate:
- Latest 3 months: 3.75%
- Previous 3 months: 4.02%


In [56]:
print(
    "========== RAG SOURCES ==========\n"
)

for i, source in enumerate(
    result["sources"],
    start=1
):

    print(
        f"SOURCE {i}"
    )

    print(
        source["metadata"]
    )

    print(
        source["document"]
    )

    print(
        "-" * 80
    )

========== RAG SOURCES ==========

SOURCE 1
{'supplier_name': 'Apex Global', 'document_type': 'review', 'date': '2025-05-31', 'supplier_id': 'SUP052'}
Supplier: Apex Global
Review Date: 2025-05-31

Performance Summary:
Supplier performance deteriorated during the recent review period. SLA compliance is 93.7%, on-time delivery is 90.0%, and defect rate is 2.7%.

Key Issues:
on-time delivery performance is below expectations. Review discussion highlighted quality-control process gaps as a likely contributing factor.

Corrective Actions:
Introduce additional process controls and monitor SLA, delivery, and defect metrics each week.

Reviewer Notes:
Priority should remain on trend monitoring and early intervention.
--------------------------------------------------------------------------------
SOURCE 2
{'date': '2025-01-31', 'document_type': 'review', 'supplier_id': 'SUP052', 'supplier_name': 'Apex Global'}
Supplier: Apex Global
Review Date: 2025-01-31

Performance Summary:
Supplier perfor